# 04 – Özellik Birleştirme (Feature Fusion)

Şimdiye kadar iki farklı yaklaşım denedik:

| Not defteri | Yöntem | Neye bakıyor? | Güçlü yanı | Zayıf yanı |
|---|---|---|---|---|
| 02 | TF-IDF | Ortak kelimeler | Bu veri setinin kelimelerinden öğreniyor | Farklı kelimelerle anlatılan doğru cevabı tanımıyor |
| 03 | Sentence-BERT | Anlam | Farklı kelimelerle anlatılan aynı fikri tanıyor | Zıt anlamlı cevapları ayırt edemiyor, öğretmen puanlarından öğrenmiyor |

İki yöntem **farklı şeyleri** yakaladığı için birbirini tamamlayabilir. Bu not defterinde şu soruya cevap arayacağız:

> **İki yöntemin özelliklerini tek bir modelde birleştirmek sonucu iyileştirir mi?**

Ayrıca birleştirme için iki farklı model deneyeceğiz: doğrusal regresyon ve gradyan artırma.

### Nasıl çalıştırılır?
Dosyayı Drive'daki `CevapAnaliz` klasörüne yükle, Colab'da aç ve **Çalışma zamanı → Tümünü çalıştır** de. GPU gerekmez. **Önce 03 numaralı not defterinin çalıştırılmış olması gerekir**, çünkü Sentence-BERT benzerliklerini oradan alıyoruz.

## 1. Kurulum

In [ ]:
import os, sys, subprocess
from pathlib import Path

COLAB = "google.colab" in sys.modules
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/CevapAnaliz")
    PROJE = Path("/content/CevapAnaliz")
    if PROJE.exists():
        subprocess.run(["git", "-C", str(PROJE), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/Semantik-Muhendisleri/CevapAnaliz.git", str(PROJE)], check=True)
    (DRIVE / "data").mkdir(parents=True, exist_ok=True)
    if not (PROJE / "data").exists():
        os.symlink(DRIVE / "data", PROJE / "data")
else:  # Kendi bilgisayarında çalıştırılırsa
    PROJE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE = PROJE

os.chdir(PROJE)
sys.path.insert(0, str(PROJE))
SEKIL = DRIVE / "sekiller"; SEKIL.mkdir(parents=True, exist_ok=True)
SONUC = DRIVE / "sonuclar"; SONUC.mkdir(parents=True, exist_ok=True)

import pandas as pd
df = pd.read_csv(PROJE / "data" / "processed" / "ortak_veri.csv", keep_default_na=False)
benzerlik_dosyasi = SONUC / "03_sbert_benzerlik.csv"
if not benzerlik_dosyasi.exists():
    raise FileNotFoundError("Sentence-BERT benzerlikleri bulunamadı. Önce 03 numaralı not defterini çalıştır.")
benzerlik = pd.read_csv(benzerlik_dosyasi)
df["MiniLM"], df["MPNet"] = benzerlik["MiniLM"].values, benzerlik["MPNet"].values
print(f"{len(df):,} cevap ve Sentence-BERT benzerlikleri yüklendi.".replace(",", "."))

## 2. Özellik birleştirme nedir?

Bir cevabı puanlarken modele verdiğimiz her sayıya **özellik** diyoruz. Elimizde artık beş özellik var:

| Özellik | Nereden geliyor? | Ne ölçüyor? |
|---|---|---|
| `kosinus` | 02 – TF-IDF (kök bulma ile) | Ortak önemli kelimeler |
| `ngram` | 02 | Referanstaki kelime gruplarının ne kadarı cevapta geçiyor |
| `uzunluk` | 02 | Cevabın referansa göre uzunluğu |
| `MiniLM` | 03 – Sentence-BERT (küçük model) | Anlam benzerliği |
| `MPNet` | 03 – Sentence-BERT (büyük model) | Anlam benzerliği |

İki yöntemi birleştirmenin iki yolu var:
- **Erken birleştirme (özellik düzeyinde):** Bütün özellikleri yan yana koyup **tek bir modele** veririz. Model hangi özelliğe ne kadar güveneceğini kendisi öğrenir.
- **Geç birleştirme (karar düzeyinde):** Her yöntem ayrı ayrı puan verir, sonra puanların ortalaması alınır.

Biz **erken birleştirmeyi** kullanıyoruz, çünkü model özellikler arasındaki ilişkiyi de öğrenebilir (örneğin "anlam benzerliği yüksek ama ortak kelime yoksa…"). 02'de üç TF-IDF özelliğini birleştirirken de aslında bunu yapmıştık.

## 3. İki birleştirme modeli

**Doğrusal regresyon** (02'den tanıdığımız): Her özelliğe bir ağırlık verir ve toplar. Basit ve anlaşılır, ama yalnızca "düz çizgi" şeklindeki ilişkileri öğrenebilir: "Benzerlik 0,1 artarsa puan hep aynı miktarda artar."

**Gradyan artırma** (gradient boosting): Çok sayıda küçük **karar ağacından** oluşur. Bir karar ağacı, evet-hayır sorularıyla karar verir:

```
Anlam benzerliği 0,7'den büyük mü?
├── Evet → Ortak kelime oranı 0,3'ten büyük mü?
│          ├── Evet → puan 0,9
│          └── Hayır → puan 0,7
└── Hayır → puan 0,3
```

Gradyan artırma bu ağaçlardan yüzlercesini **sırayla** kurar; her yeni ağaç, öncekilerin yaptığı hataları düzeltmeye çalışır. Böylece "benzerlik belli bir değerin üstüne çıkınca puan hızla artar" gibi **eğri** ilişkileri de öğrenebilir. scikit-learn kütüphanesindeki hazır `HistGradientBoostingRegressor` modelini kullanıyoruz.

Karşılaştıracağımız beş yöntem:

| Yöntem | Özellikler | Model |
|---|---|---|
| TF-IDF · Doğrusal | kosinus, ngram, uzunluk | Doğrusal regresyon (02'nin en iyisi) |
| TF-IDF + MiniLM · Doğrusal | + MiniLM | Doğrusal regresyon |
| TF-IDF + SBERT · Doğrusal | + MiniLM + MPNet | Doğrusal regresyon |
| TF-IDF · Gradyan artırma | kosinus, ngram, uzunluk | Gradyan artırma |
| TF-IDF + SBERT · Gradyan artırma | beş özelliğin hepsi | Gradyan artırma |

In [ ]:
import re
import numpy as np
from nltk.stem import PorterStemmer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold
from src.puanlama.ozellikler import TfidfKosinus, ozellik_tablosu
from src.puanlama.olcutler import olc

# 02'deki en iyi TF-IDF ayarı: kök bulma
_kok_bulucu, _onbellek, _KELIME = PorterStemmer(), {}, re.compile(r"[a-z0-9]+")
def kokle(metin):
    sonuc = []
    for k in _KELIME.findall(metin.lower().replace("n't", " not")):
        if k not in _onbellek:
            _onbellek[k] = _kok_bulucu.stem(k)
        sonuc.append(_onbellek[k])
    return " ".join(sonuc)

kok_df = df.copy()
kok_df["referans_cevap"] = kok_df["referans_cevap"].map(kokle)
kok_df["ogrenci_cevap"] = kok_df["ogrenci_cevap"].map(kokle)

def ozellikler(egitim_i, test_i):
    """TF-IDF özelliklerini eğitim verisinden öğrenip Sentence-BERT benzerlikleriyle yan yana koyar."""
    egitim, test = kok_df.loc[egitim_i], kok_df.loc[test_i]
    kosinus = TfidfKosinus().ogren(pd.concat([egitim["referans_cevap"], egitim["ogrenci_cevap"]]))
    oz_e, oz_t = ozellik_tablosu(egitim, kosinus), ozellik_tablosu(test, kosinus)
    for sutun in ["MiniLM", "MPNet"]:
        oz_e[sutun], oz_t[sutun] = df.loc[egitim_i, sutun].values, df.loc[test_i, sutun].values
    return oz_e, oz_t

TFIDF = ["kosinus", "ngram", "uzunluk"]
YONTEMLER = {
    "TF-IDF · Doğrusal": (TFIDF, "dogrusal"),
    "TF-IDF + MiniLM · Doğrusal": (TFIDF + ["MiniLM"], "dogrusal"),
    "TF-IDF + SBERT · Doğrusal": (TFIDF + ["MiniLM", "MPNet"], "dogrusal"),
    "TF-IDF · Gradyan artırma": (TFIDF, "gradyan"),
    "TF-IDF + SBERT · Gradyan artırma": (TFIDF + ["MiniLM", "MPNet"], "gradyan"),
}

def yeni_model(tur):
    if tur == "dogrusal":
        return LinearRegression()
    return HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=0)

def testler():
    mohler_i = df.index[df["veri_seti"] == "mohler"]
    for egitim, test in GroupKFold(n_splits=5).split(mohler_i, groups=df.loc[mohler_i, "soru_id"]):
        yield "Mohler", mohler_i[egitim], mohler_i[test], 5
    for veri_seti, ad in [("scientsbank", "SciEntsBank"), ("beetle", "Beetle")]:
        egitim = df.index[(df["veri_seti"] == veri_seti) & (df["bolum"] == "train")]
        for bolum, aciklama in [("test_ua", "görülmemiş cevaplar"), ("test_uq", "görülmemiş sorular")]:
            yield f"{ad} – {aciklama}", egitim, df.index[(df["veri_seti"] == veri_seti) & (df["bolum"] == bolum)], 2

tahminler = {y: pd.Series(np.nan, index=df.index) for y in YONTEMLER}
test_listesi = {}
for test_adi, egitim_i, test_i, seviye in testler():
    test_listesi.setdefault(test_adi, (seviye, []))[1].append(test_i)
    oz_e, oz_t = ozellikler(egitim_i, test_i)
    for yontem, (sutunlar, tur) in YONTEMLER.items():
        model = yeni_model(tur).fit(oz_e[sutunlar], df.loc[egitim_i, "puan"])
        tahminler[yontem].loc[test_i] = np.clip(model.predict(oz_t[sutunlar]), 0, 1)

satirlar = []
for test_adi, (seviye, parcalar) in test_listesi.items():
    test_i = np.concatenate(parcalar)
    for yontem in YONTEMLER:
        satirlar.append({"test": test_adi, "yontem": yontem,
                         **olc(df.loc[test_i, "puan"], tahminler[yontem].loc[test_i], seviye)})
sonuc = pd.DataFrame(satirlar)
sonuc.round(4).to_csv(SONUC / "04_ozellik_birlestirme.csv", index=False, encoding="utf-8")

TEST_SIRA = list(test_listesi)
for olcut in ["pearson", "qwk"]:
    print(f"\n{olcut.upper()}")
    display(sonuc.pivot(index="yontem", columns="test", values=olcut).loc[list(YONTEMLER), TEST_SIRA].round(3))

In [ ]:
import matplotlib.pyplot as plt

GOSTER = {"TF-IDF · Doğrusal": "#2a78d6", "TF-IDF + SBERT · Doğrusal": "#eb6834",
          "TF-IDF + SBERT · Gradyan artırma": "#1baf7a"}
fig, eksenler = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True)
genislik = 0.26
for ax, olcut, baslik in zip(eksenler, ["pearson", "qwk"], ["Pearson korelasyonu", "QWK"]):
    for i, (yontem, renk) in enumerate(GOSTER.items()):
        degerler = sonuc[sonuc["yontem"] == yontem].set_index("test").loc[TEST_SIRA, olcut]
        ax.bar(np.arange(len(TEST_SIRA)) + (i - 1) * genislik, degerler, width=genislik - 0.03,
               color=renk, label=yontem)
    ax.axhline(0.70, color="#52514e", linewidth=1, linestyle="--")
    ax.set_xticks(np.arange(len(TEST_SIRA)), [t.replace(" – ", "\n") for t in TEST_SIRA], fontsize=8.5)
    ax.set_title(baslik, loc="left", fontsize=12)
    ax.set_ylim(0, 0.8)
    for kenar in ["top", "right"]:
        ax.spines[kenar].set_visible(False)
    ax.grid(axis="y", color="#e6e5e0"); ax.set_axisbelow(True)
eksenler[0].text(len(TEST_SIRA) - 0.5, 0.71, "hedef 0,70", ha="right", va="bottom", fontsize=9, color="#52514e")
tutamaklar, etiketler = eksenler[0].get_legend_handles_labels()
fig.legend(tutamaklar, etiketler, frameon=False, loc="upper left", ncol=3, bbox_to_anchor=(0.01, 0.93), fontsize=10)
fig.suptitle("Özellik birleştirmenin etkisi", x=0.01, ha="left", fontsize=14, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(SEKIL / "04_ozellik_birlestirme.png", dpi=150, bbox_inches="tight")
plt.show()

**Ne görüyoruz?**

**1. Sentence-BERT'i eklemek küçük ama tutarlı bir iyileşme sağlıyor.** Doğrusal regresyonda TF-IDF'e Sentence-BERT benzerliklerini ekleyince:
- **Pearson** 5 testin 4'ünde arttı. En büyük artış Mohler'de: 0,54 → 0,57. Sadece Beetle'ın görülmemiş sorularında düştü (0,34 → 0,32).
- **QWK** 5 testin **hepsinde** arttı: Mohler 0,36 → 0,41, Beetle (görülmemiş cevaplar) 0,23 → 0,37.

03'te tek başına TF-IDF'in gerisinde kalan Sentence-BERT, TF-IDF'le birlikte kullanıldığında işe yarıyor: Kelime tabanlı yöntemin kaçırdığı bir şeyi yakalıyor.

**2. MiniLM eklemek, iki modeli birden eklemekle neredeyse aynı sonucu veriyor.** MPNet'in ek katkısı çok az; iki model büyük ölçüde aynı bilgiyi taşıyor.

**3. Gradyan artırma QWK'da en iyisi, Pearson'da değil.** Gradyan artırma QWK'yı belirgin şekilde yükseltti (Mohler'de 0,47, Beetle'da 0,44; testlerin çoğunda en yüksek değer), ama Pearson'da doğrusal regresyonla aynı düzeyde ya da biraz gerisinde kaldı (Mohler'de 0,53'e karşı 0,57). Bunun sebebini bir sonraki bölümde göreceğiz.

**4. Hâlâ hedefin uzağındayız.** En iyi Pearson 0,57, en iyi QWK 0,47. Hedef ikisi için de 0,70.

## 4. Neden gradyan artırmada QWK bu kadar yükseldi?

QWK, puanları basamaklara yuvarlayıp karşılaştırır (Mohler'de 0–5, SemEval'de yanlış / kısmen doğru / doğru). Doğrusal regresyonun bilinen bir eğilimi vardır: Emin olamadığında tahminleri **ortalamaya yakın** tutar. Bu yüzden çok iyi ve çok kötü cevapları da "orta" puana yuvarlar. Tahminlerin ne kadar yayıldığına bakalım:

In [ ]:
dagilim = []
for test_adi, (seviye, parcalar) in test_listesi.items():
    test_i = np.concatenate(parcalar)
    satir = {"test": test_adi, "Öğretmen puanları": df.loc[test_i, "puan"].std()}
    for yontem in ["TF-IDF + SBERT · Doğrusal", "TF-IDF + SBERT · Gradyan artırma"]:
        satir[yontem] = tahminler[yontem].loc[test_i].std()
    dagilim.append(satir)
print("Puanların standart sapması (ne kadar yayıldıkları; büyük = daha yayılmış):")
display(pd.DataFrame(dagilim).set_index("test").round(3))

**Ne görüyoruz?**

- Öğretmen puanlarının standart sapması Mohler'de 0,22, SciEntsBank ve Beetle'da yaklaşık 0,44.
- Doğrusal regresyonun tahminleri bunun **yaklaşık yarısı kadar** yayılıyor (SemEval'de 0,18–0,21; Mohler'de de benzer bir daralma var). Yani model çok iyi ve çok kötü cevapları da ortaya doğru çekiyor; tahminlerin çoğu "orta" puanda toplanıyor.
- Gradyan artırmanın tahminleri **daha yayılmış** (SemEval'de 0,22–0,25). Daha fazla cevap "doğru" ya da "yanlış" basamağına ulaştığı için QWK yükseliyor.

İki model de öğretmenden daha **çekingen**: Uç puanlar vermekten kaçınıyorlar. Bu, elimizdeki özelliklerin doğru cevabı yanlıştan kesin olarak ayırmaya yetmediğinin bir işareti. Model emin olamadığında güvenli tarafta kalıp ortalamaya yakın puan veriyor.

**Hangisini seçmeliyiz?** QWK, öğretmenin göreceği **yuvarlanmış puanların** ne kadar doğru olduğunu ölçtüğü için sistemimiz açısından daha anlamlı ölçüt. Bu yüzden klasik yöntemler arasında **TF-IDF + Sentence-BERT özellikleriyle gradyan artırma** en iyi seçenek.

## 5. Model hangi özelliğe güveniyor?

Bunu anlamak için basit bir deney yapıyoruz: Bir özelliğin değerlerini cevaplar arasında **rastgele karıştırıp** modelin başarısının ne kadar düştüğüne bakıyoruz. Bir özellik önemliyse, karıştırılınca model şaşırır ve Pearson belirgin şekilde düşer. Önemsizse neredeyse hiçbir şey değişmez. Bu yönteme **permütasyon önemi** denir.

En büyük veri setimiz olan SciEntsBank'te (görülmemiş cevaplar), beş özelliğin hepsini kullanan gradyan artırma modeline bakıyoruz.

In [ ]:
from scipy.stats import pearsonr

egitim_i = df.index[(df["veri_seti"] == "scientsbank") & (df["bolum"] == "train")]
test_i = df.index[(df["veri_seti"] == "scientsbank") & (df["bolum"] == "test_ua")]
oz_e, oz_t = ozellikler(egitim_i, test_i)
TUM = TFIDF + ["MiniLM", "MPNet"]
model = yeni_model("gradyan").fit(oz_e[TUM], df.loc[egitim_i, "puan"])
gercek = df.loc[test_i, "puan"].values
temel = pearsonr(gercek, model.predict(oz_t[TUM]))[0]

rastgele = np.random.default_rng(0)
onem = {}
for sutun in TUM:
    dususler = []
    for _ in range(10):
        karisik = oz_t[TUM].copy()
        karisik[sutun] = rastgele.permutation(karisik[sutun].values)
        dususler.append(temel - pearsonr(gercek, model.predict(karisik))[0])
    onem[sutun] = np.mean(dususler)
onem = pd.Series(onem).sort_values()

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(onem.index, onem.values, color="#2a78d6", height=0.6)
for y, deger in enumerate(onem.values):
    ax.text(deger + 0.003, y, f"{deger:.3f}".replace(".", ","), va="center", fontsize=9)
ax.set_xlabel("Özellik karıştırılınca Pearson'daki düşüş")
for kenar in ["top", "right"]:
    ax.spines[kenar].set_visible(False)
ax.set_title(f"Özelliklerin önemi (SciEntsBank, temel Pearson {temel:.2f})".replace(".", ","),
             loc="left", fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(SEKIL / "04_ozellik_onemi.png", dpi=150, bbox_inches="tight")
plt.show()

**Ne görüyoruz?**

- **TF-IDF kosinüs benzerliği açık ara en önemli özellik:** Karıştırılınca Pearson 0,31 düşüyor; yani model neredeyse hiçbir şey bilemez hâle geliyor.
- **MPNet ikinci sırada** (0,05): Sentence-BERT'in gerçekten ek bir bilgi taşıdığını burada da görüyoruz.
- Uzunluk oranı (0,03) ve n-gram örtüşmesi (0,02) küçük katkı yapıyor.
- **MiniLM neredeyse önemsiz görünüyor** (0,005). Bu, MiniLM'in işe yaramadığı anlamına gelmiyor: MPNet ile neredeyse aynı bilgiyi taşıdığı için, MiniLM karıştırıldığında model MPNet'ten aynı bilgiyi almaya devam ediyor. Birbirine çok benzeyen özellikler önemi aralarında paylaşır; bu permütasyon öneminin bilinen bir özelliğidir.

Sonuç: Model puanı tahmin ederken **en çok kelime benzerliğine** güveniyor; anlam benzerliği **ikinci derecede** yardımcı oluyor.

## Özet

| Bulgu | Sonucu |
|---|---|
| TF-IDF'e Sentence-BERT eklemek Pearson'ı 5 testin 4'ünde, QWK'yı hepsinde artırdı | İki yöntem birbirini tamamlıyor; özellik birleştirme işe yarıyor |
| En büyük artış Mohler'de: Pearson 0,54 → 0,57, QWK 0,36 → 0,41 | İyileşme küçük ama tutarlı |
| Gradyan artırma QWK'da en iyisi (Mohler'de 0,47'ye kadar) | Tahminleri daha geniş yaydığı için yuvarlanmış puanlar daha doğru |
| Model en çok TF-IDF kosinüsüne, ikinci olarak MPNet'e güveniyor | Anlam benzerliği yardımcı bilgi olarak katkı yapıyor |
| En iyi Pearson 0,57, en iyi QWK 0,47 | Hedef (0,70) hâlâ uzakta |

**Klasik yöntemlerden en iyisi:** TF-IDF + Sentence-BERT özellikleri ile gradyan artırma.

**Neden hâlâ hedefin uzağındayız?** Bu not defterindeki bütün özellikler, iki metin arasındaki benzerliği **dışarıdan ölçen** sayılar. Model metni kendisi okumuyor; "denser" ile "less dense" arasındaki zıtlık gibi ayrıntılar bu sayılarda kayboluyor.

Sonuçlar Drive'da `CevapAnaliz/sonuclar/04_ozellik_birlestirme.csv`, grafikler `CevapAnaliz/sekiller/04_ozellik_birlestirme.png` ve `04_ozellik_onemi.png` dosyalarına kaydedildi.

**Sonraki not defteri: 05 – İnce Ayar.** Referans cevabı ve öğrenci cevabını **birlikte okuyan** bir dil modelini öğretmen puanlarıyla eğiteceğiz. Model artık hazır benzerlik sayılarına değil, metnin kendisine bakacak.